In [ ]:
# # Code to convert this notebook to .py (if you want to run it via command line or with Slurm)
# from subprocess import call
# command = "jupyter nbconvert evaluate.ipynb --to python"
# call(command,shell=True)

## Import Libraries/Packages/Functions

In [ ]:
import os
import argparse

# local
import utils

## Configurations

In [ ]:
if utils.is_interactive():
    # sample usage
    jupyter_args = "--data_repo_id=potsu-potsu/mini-bioasq-with-metadata \
                    --file=data/retrieved-docs/qdrant-dense-bioasq-snowflake-multi.json \
                    --files_list=data/retrieved-docs/okapi-bm25.json,data/retrieved-docs/qdrant-dense-bioasq-med-cpt.json,data/retrieved-docs/qdrant-dense-bioasq-snowflake.json,data/retrieved-docs/qdrant-bioasq-snowflake-hybrid.json,data/retrieved-docs/qdrant-dense-bioasq-snowflake-multi.json,data/retrieved-docs/qdrant-dense-bioasq-snowflake-keyword-filtered.json \
                    --run_names=Okapi-BM25,MedCPT,Snowflake,Snowflake-BM25,Snowflake-MRL,Snowflake-MRL-Keywords \
                    --p_value=0.05"
    
    jupyter_args = jupyter_args.split()
    print(jupyter_args)

    %load_ext autoreload 
    %autoreload 2

In [ ]:
def list_of_strings(arg):
    return arg.split(',')

parser = argparse.ArgumentParser(description="Document Retrieval Evaluation")
parser.add_argument(
    "--data_repo_id", type=str,
    help="HuggingFace repository ID where dataset is stored",
)
parser.add_argument(
    "--cache_dir", type=str, default=None,
    help="path to the folder where cached files are stored; if not provided, HuggingFace will use the default cache_dir",
)
parser.add_argument(
    "--local_dir", type=str, default=None,
    help="if provided, the downloaded file will be placed under this directory",
)
parser.add_argument(
    "--file", type=str,
    help="run file in JSON format",
)
parser.add_argument(
    "--metrics", type=list_of_strings,
    help="evaluation metrics",
)
parser.add_argument(
    "--eval_results_dir", type=str, default="data/eval-results",
    help="path where retrieved documents will be saved",
)
parser.add_argument(
    "--eval_results_filename", type=str, default=None,
    help="filename used when saving evaluation results",
)
parser.add_argument(
    "--files_list", type=list_of_strings,
    help="list of run files to compare results",
)
parser.add_argument(
    "--run_names", type=list_of_strings,
    help="list of run names to use when comparing results",
)
parser.add_argument(
    "--stat_test", type=str, default="fisher", choices=['student', 'fisher', 'tukey'],
    help="statistical test to use",
)
parser.add_argument(
    "--p_value", type=float, default=0.05,
    help="p-value threshold",
)


if utils.is_interactive():
    args = parser.parse_args(jupyter_args)
else:
    args = parser.parse_args()

# create global variables without the args prefix
for attribute_name in vars(args).keys():
    globals()[attribute_name] = getattr(args, attribute_name)


for arg_name, arg_value in vars(args).items():
    print(f"{arg_name}: {arg_value}")



In [ ]:
os.makedirs(eval_results_dir, exist_ok=True)

In [ ]:
if not metrics:
    metrics = [
        'ndcg@3',
        'ndcg@5',
        'ndcg@10',
        'precision@3',
        'precision@5',
        'precision@10',
        'recall@3',
        'recall@5',
        'recall@10', 
    ]

## Load Data

In [ ]:
test_file = "question-answer-passages/test-00000-of-00001.parquet"

In [ ]:
qrels = utils.get_eval_qrels_huggingface(
    repo_id=data_repo_id, filename=test_file, cache_dir=cache_dir, local_dir=local_dir
)

qrels

## Evaluate

In [ ]:
file

In [ ]:
eval_results, df = utils.evaluate_retrieval(file=file, qrels_dict=qrels, metrics=metrics, return_mean=True)
df

In [ ]:
if eval_results_filename is None:
    filename = os.path.splitext(os.path.basename(file))[0]
    eval_results_filename = f'{filename}-eval'

save_eval_dir = f"{eval_results_dir}/{eval_results_filename}"
save_eval_dir

In [ ]:
df_results.to_csv(f"{save_eval_dir}.csv")

## Statistical Significance

In [ ]:
## no correction
report = utils.compare_evaluations(
    files=files_list, 
    run_names=run_names, 
    qrels_dict=qrels, 
    metrics=metrics, 
    stat_test=stat_test, 
    max_p=p_value)

In [ ]:
report 

In [ ]:
report, all_stats, all_pairwise_stats  = utils.compare_evaluations_stats(
    files=files_list, 
    run_names=run_names, 
    qrels_dict=qrels, 
    metrics=metrics, 
    stat_test=stat_test, 
    max_p=p_value, 
    correction="holm-bonf"
)

In [ ]:
report

In [ ]:
df = report.to_dataframe()
df

In [ ]:
save_eval_dir = f"{eval_results_dir}/report"

In [ ]:
df.to_csv(f"{save_eval_dir}.csv")

## Plot

In [ ]:
utils.plot_forest(all_pairwise_stats=all_pairwise_stats, 
            base_model="Snowflake-MRL", 
            metrics_directions={
                'ndcg@10': True, 
                'precision@10': True, 
                'recall@10': True}
            )

In [ ]:
utils.plot_forest(all_pairwise_stats=all_pairwise_stats, 
            base_model="Snowflake-MRL",
            metrics_directions={'time_seconds': False})